In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
from contextlib import contextmanager
import torch
import torch.nn as nn
from tqdm import tqdm
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random

from common import *

import library

In [ ]:
set_seed(0)
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = ["blue", "red", "green"]
plt.rcParams.update({
    'text.usetex': False,
    # 'font.family': 'serif',
    # 'font.serif': ['Computer Modern Roman', 'CMU Serif', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',  # Computer Modern math font
})
# plt.rcParams.update({
#     "text.usetex": True,
#     'font.family': 'serif',
#     'text.latex.preamble': r"\usepackage{amsmath}",
#     # "axes.formatter.use_mathtext" : True,
# })

default plot setting

In [ ]:
def sample(batch, sizes, device, mul=0.01, alphas=[1, 1]):
    x_out = torch.zeros((batch, 0), device=device)
    for size, alpha in zip(sizes, alphas):
        rand = alpha * torch.rand((batch, 1), device=device).round() * 2 - 1
        x_out = torch.cat([x_out, rand.repeat(1, size)], dim=-1)
    epsilon1 = mul * torch.randn_like(x_out)
    epsilon2 = mul * torch.randn_like(x_out)
    return x_out + epsilon1, x_out + epsilon2

sample makes [+1, -1] datas

In [ ]:
class BTLoss(nn.Module):
    def __init__(self, in_feature, lambd=1):
        super().__init__()
        # self.bn = nn.BatchNorm1d(in_feature, affine=False)
        self.bn = nn.Identity()
        self.lambd = lambd

    def forward(self, x0, x1):
        c = self.bn(x0).T @ self.bn(x1)
        c = c + c.T
        c.div_(x0.shape[0] * 2)
        output = c.clone()
        on_diag = torch.diagonal(c).add_(-1).pow_(2).sum()
        off_diag = off_diagonal(c).pow_(2).sum()
        loss = on_diag + self.lambd * off_diag
        return output, loss


def off_diagonal(x):
    # return a flattened view of the off-diagonal elements of a square matrix
    n, m = x.shape
    assert n == m
    return x.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()

In [ ]:
class SimCLRLoss(nn.Module):
    def __init__(self, batch_size, temperature):
        super().__init__()
        # self.bn = nn.BatchNorm1d(in_feature, affine=False)
        self.bn = nn.Identity()
        self.batch_size = batch_size
        self.temperature = temperature

        self.mask = self.mask_correlated_samples(batch_size)
        self.criterion = nn.CrossEntropyLoss(reduction="sum")
        self.similarity_f = nn.CosineSimilarity(dim=2)

    def mask_correlated_samples(self, batch_size):
        N = 2 * batch_size
        mask = torch.ones(size=(N, N), dtype=torch.bool)
        mask = mask.fill_diagonal_(0)

        for i in range(batch_size):
            mask[i, batch_size + i] = 0
            mask[batch_size + i, i] = 0
        return mask

    def forward(self, z_i, z_j):
        c = self.bn(z_i).T @ self.bn(z_j)
        c = c + c.T
        c.div_(x0.shape[0] * 2)

        N = 2 * self.batch_size

        z = torch.cat((z_i, z_j), dim=0)

        sim = self.similarity_f(z.unsqueeze(1), z.unsqueeze(0)) / self.temperature

        sim_i_j = torch.diag(sim, self.batch_size)
        sim_j_i = torch.diag(sim, -self.batch_size)

        positive_samples = torch.cat((sim_i_j, sim_j_i), dim=0).reshape(N, 1)
        negative_samples = sim[self.mask].reshape(N, -1)

        labels = torch.from_numpy(np.array([0]*N)).reshape(-1).to(positive_samples.device).long()

        logits = torch.cat((positive_samples, negative_samples), dim=1)
        loss = self.criterion(logits, labels)
        loss /= N

        return c, loss

In [ ]:
@contextmanager
def scaled_model(model, scale):
    assert scale != 0
    try:
        scale_weights(model, scale)
        yield model
    finally:
        scale_weights(model, 1 / scale)

@torch.no_grad()
def scale_weights(model, scale: float = 1) -> None:
    for name, param in model.named_parameters():
        if "weight" in name and scale != 1:
            transformed_param = param * scale
            param.copy_(transformed_param)

In [ ]:
def create_segment_vectors(sizes):
    total_length = sum(sizes)
    cumsum = torch.cumsum(torch.tensor(sizes), dim=-1)

    result = torch.zeros((len(sizes), total_length))

    for i in range(len(sizes)):
        start_idx = cumsum[i - 1] if i > 0 else 0
        end_idx = cumsum[i]

        result[i, start_idx:end_idx] = 1

    return result

In [ ]:
batch = 1000
epochs = 4000

sizes = [9, 1]
alphas = [1, 1]
s_in = sum(sizes)
s_out = 2
device = torch.device("cuda")
simple_network = nn.Linear(s_in, s_out, bias=False, device=device)
lr_simple = 1e-3
scaling = 1e-4
with torch.no_grad():
    torch.nn.init.normal_(simple_network.weight)
print(torch.var(simple_network.weight), torch.mean(simple_network.weight))

loss_type = "BarlowTwins"
if loss_type == "BarlowTwins":
    lossFn = BTLoss(s_out, lambd=1).to("cuda")
else:
    lossFn = SimCLRLoss(batch, 0.1).to("cuda")
t = sample(batch, sizes, device, alphas=alphas)
gamma = torch.bmm(t[0].unsqueeze(dim=2), t[1].unsqueeze(dim=1))  # b x n x n
gamma = gamma + gamma.permute(0, 2, 1)
gamma = torch.mean(gamma, dim=0).squeeze() / 2  # n x n
print(gamma)
plt.imshow(gamma.cpu())
plt.show()
gamma_l, gamma_v = torch.linalg.eig(gamma)
gamma_l = gamma_l.real.cpu()
gamma_v = gamma_v.real.cpu()
print(gamma_l, gamma_v)

scale_weights(simple_network, scaling)
w = simple_network.weight.detach().clone()
u, s0, g = torch.linalg.svd(w)
plt.imshow(gamma_l.unsqueeze(dim=0))
plt.show()
plt.imshow(gamma_v)
plt.show()
optimizer = torch.optim.SGD(simple_network.parameters(), lr_simple, momentum=0, weight_decay=0)
simple_network.train()

loss_values = []
eiglist = [[] for i in range(s_out)]
we = [[] for _ in range(len(alphas))]
e = create_segment_vectors(sizes).to(device)
print("e", e.shape)
weight_singular = [[] for _ in range(len(alphas))]
plane_alignment = []
uwe = [[[] for _ in range(2)] for _ in range(2)]

eig_reversed = False
dist = [[] for i in range(2)]
for epoch in tqdm(range(1, epochs + 1)):
    optimizer.zero_grad()
    x0, x1 = sample(batch, sizes, device, alphas=alphas, mul=0.01)
    z0, z1 = torch.chunk(simple_network(torch.cat([x0.clone(), x1.clone()], dim=0)), 2, dim=0)
    c, loss = lossFn(z0, z1)
    c = c.detach()
    if epoch == 1:
        print("output", c)
    v = reversed(torch.linalg.eigvalsh(c))
    for i in range(len(v)):
        eiglist[i].append(v[i].cpu().item())
    loss_values.append(loss.item())
    loss.backward()
    optimizer.step()
    
    u_prev = u_w_current if epoch > 1 else None
    w_current = simple_network.weight.detach().clone()
    u_w_current, s_w_current, v_w_current_h = torch.linalg.svd(w_current)
    v_w_current = v_w_current_h.T
    
    if epoch > 1:
        dist1 = abs(F.cosine_similarity(u_prev[:, 0], u_w_current[:, 0], dim=0))
        dist2 = abs(F.cosine_similarity(u_prev[:, 1], u_w_current[:, 0], dim=0))
        dist[0].append(dist1.cpu().item())
        dist[1].append(dist2.cpu().item())
        eig_reversed = dist1 < dist2
    if eig_reversed:
        u_w_current = u_w_current[:, [1, 0]]
        s_w_current = s_w_current[[1, 0]]
        v_w_current = v_w_current[:, [1, 0]]
    u_prev = u_w_current.clone()

    # Since PyTorch's eigenvalues are sorted by magnitude, the order can be reversed from the eigenvector's perspective.

    for i in range(len(alphas)):
        weight_singular[i].append(s_w_current[i].cpu().item())
        we[i].append(torch.norm(e[i].unsqueeze(dim=0) @ w_current.T).cpu().item())
    plane_alignment.append((torch.norm(v_w_current[:, :2].T @ gamma_v[:, :2].cuda(), p='fro')**2/2).cpu().item())
    
    for i in range(2):
        for j in range(2):
            uwe[i][j].append(((u_w_current[:, i] @ w_current @ e[j].cuda()).cpu()).item())
            

for i in range(2):
    for j in range(2):
        uwe[i][j] = np.array(uwe[i][j])

In [ ]:
def dt(input):
    output = []
    for i in range(1, len(input)):
        output.append(input[i] - input[i - 1])
    return np.array(output)

In [ ]:
def smooth(y, box_pts):
    box = np.ones(box_pts)/box_pts
    y_smooth = np.convolve(y, box, mode='same')
    return y_smooth

In [ ]:
gamma_0, gamma_1 = gamma_l[0], gamma_l[1]
t0 = 1030
t = np.linspace(0, epochs, epochs)
y1 = gamma_0 / (gamma_0 + s0[0].cpu()**(-2) * np.exp(-8 * gamma_0 * t/t0))
y2 = gamma_1 / (gamma_1 + s0[1].cpu()**(-2) * np.exp(-8 * gamma_1 * t/t0))
lll = (y1-1)**2 + (y2-1)**2

In [ ]:
palette = plt.rcParams['axes.prop_cycle'].by_key()['color']
alpha = 0.8
fig, axs = plt.subplots(1, 3, figsize=(24, 3.7))

axs[0].plot(loss_values, color=colors[2], linewidth=6, alpha=0.5)
index = []
for i in range(len(eiglist)):
    index.append(((torch.tensor(loss_values) - (s_out - i - 0.5)) < 0).nonzero()[0])


axs[0].plot(lll, linestyle='--', color=colors[2], linewidth=2.5)
axs[0].set_ylim(-0.05, s_out * 1.15)
print(index)
axs[0].vlines(index, -1, 3, colors='black', linestyle='--')
axs[0].set_xlabel("Step")
axs[0].set_ylabel("Loss")

for i in range(len(eiglist)):
    axs[1].plot(eiglist[i], linewidth=6, color=colors[i], alpha=0.5, label=(r"$\lambda_" + ("l$" if i == 0 else "s$")))
axs[1].plot(t, y1, linestyle='--', color=colors[0], linewidth=2.5)
axs[1].plot(t, y2, linestyle='--', color=colors[1], linewidth=2.5)
axs[1].vlines(index, -1, 3, colors='black', linestyle='--')
axs[1].set_ylim(-0.05, 1.10)
axs[1].set_xlabel("Step")
axs[1].set_ylabel(r"C Eigenvalue $\lambda$")
axs[1].legend()

for i in range(len(we)):
    k = we[i]
    # for j in range(len(k)):
    #     k[j] /= k[-1]
    axs[2].plot(k, label=(r"${\text{FA}(e_l)}$" if i == 0 else r"${\text{FA}(e_s)}$"), alpha=0.5, linewidth=6, color=colors[i])
axs[2].plot(t, np.sqrt(y1), linestyle='--', color=colors[0], linewidth=2.5)
axs[2].plot(t, np.sqrt(y2), linestyle='--', color=colors[1], linewidth=2.5)
axs[2].vlines(index, -1, 3, colors='black', linestyle='--')
axs[2].set_ylim(-0.05, 1.10)
axs[2].legend()
axs[2].set_xlabel("Step")
axs[2].set_ylabel(r"$\|We_i\|_2$")


plt.tight_layout()
plt.savefig("figure/dim_loss_eig_align.pdf", format="pdf", bbox_inches = 'tight')
plt.show()

In [ ]:
def smooth_data(data, window_size=5):
    data_array = np.array(data)
    smooth_data = np.copy(data_array)
    half_window = window_size // 2
    for i in range(len(data_array)):
        start_idx = max(0, i - half_window)
        end_idx = min(len(data_array), i + half_window + 1)
        smooth_data[i] = np.mean(data_array[start_idx:end_idx])
    return smooth_data

In [ ]:
#dc/dt, -dl/dt
fig, axs = plt.subplots(1, 1, figsize=(14, 3.7))
for i in range(2):
    plt.plot(smooth_data(np.array(dt(eiglist[i])) / lr_simple, window_size=10), alpha=0.5, linewidth=5, color=colors[i], label=r"$d\lambda_" + ("l" if i == 0 else "s") + "/dt$")

dlossdt = -1 * np.array(dt(loss_values))
axs.plot(dlossdt / lr_simple, alpha=1, linestyle='--', color='black', linewidth=1.5, label=r"$-d\mathcal{L}/dt$")
# axs.hlines([max(dlossdt) / lr_simple], xmin=-1, xmax=epochs*1.1, color='black', alpha=0.8, linewidth=3)
# axs.hlines([max(dlossdt[1000:]) / lr_simple], xmin=-1, xmax=epochs*1.1, color='black', alpha=0.8, linewidth=3)
axs.hlines([gamma_0 * 2], xmin=-1, xmax=epochs*1.1, color=colors[0], linewidth=3, alpha=0.8, linestyle='--')
axs.hlines([gamma_1 * 2], xmin=-1, xmax=epochs*1.1, color=colors[1], linewidth=3, alpha=0.8, linestyle='--')

axs.set_yticks([0, 10, 20,])

y_pos = [text.get_position()[1] for text in axs.get_yticklabels()]#[:-1]
y_txt = [text.get_text() for text in axs.get_yticklabels()]#[:-1]

new_pos0 = gamma_0 * 2
new_txt0 = r"$2\gamma_l$"
new_pos1 = gamma_1 * 2
new_txt1 = r"$2\gamma_s$"

y_pos.append(new_pos0)
y_pos.append(new_pos1)
y_txt.append(new_txt0)
y_txt.append(new_txt1)
axs.set_yticks(y_pos,y_txt)

axs.set_ylim(-0.5, gamma_0 * 2 * 1.2)
axs.set_xlim(0, epochs)
axs.set_xlabel("Step")
# axs.set_ylabel("Value")
axs.legend(loc='upper right')
plt.savefig("figure/dim_dcdt_dldt.pdf", format="pdf", bbox_inches = 'tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 4))
# t-s_j
for i in range(2):
    lambda_i = y1 if i == 0 else y2
    gamma_i = gamma_l[i].item()
    axs[0].plot(weight_singular[i], alpha=0.5, linewidth=6, color=colors[i], label=(r"$s_" + ("l$" if i == 0 else "s$")))
    
    s_j = np.power(lambda_i / gamma_i, 1/2)
    axs[0].plot(s_j, alpha=0.8, linewidth=4, color=colors[i], linestyle='--')
    a_j = np.power(s_j, -1/2)
    p = (s_j[0]**(-2))*np.exp(-8*gamma_i*t/t0)
    s_j_p_theory = 1/(2*s_j)*(-np.power(gamma_i + p, -2) * (-8 * gamma_i * p))
    axs[1].plot(s_j_p_theory, alpha=0.8, linewidth=4, color=colors[i], linestyle='--')
print(weight_singular[0][-1])
print(weight_singular[1][-1])
axs[0].set_xlim(0, epochs)
axs[0].hlines([1/np.sqrt(gamma_0)], xmin=-1, xmax=epochs*1.1, color=colors[0], linewidth=3, alpha=0.8, linestyle='--')
axs[0].hlines([1/np.sqrt(gamma_1)], xmin=-1, xmax=epochs*1.1, color=colors[1], linewidth=3, alpha=0.8, linestyle='--')
axs[0].set_xlabel("Step")
axs[0].set_ylabel(r"$s_j$")

axs[0].set_yticks([0, 0.6, 1.2,])
y_pos = [text.get_position()[1] for text in axs[0].get_yticklabels()]#[:-1]
y_txt = [text.get_text() for text in axs[0].get_yticklabels()]#[:-1]

new_pos0 = 1/np.sqrt(gamma_0)
new_txt0 = r"$1/\sqrt{\gamma_l}$"
new_pos1 = 1/np.sqrt(gamma_1)
new_txt1 = r"$1/\sqrt{\gamma_s}$"

y_pos.append(new_pos0)
y_pos.append(new_pos1)
y_txt.append(new_txt0)
y_txt.append(new_txt1)
axs[0].set_yticks(y_pos,y_txt)

#t-ds_j/dt
for i in range(2):
    axs[1].plot(np.array(dt(weight_singular[i])) / lr_simple, alpha=0.5, linewidth=6, color=colors[i], label=(r"$\frac{ds_" + ("l" if i == 0 else "s") + r"}{dt}$"))
axs[1].set_xlim(0, epochs)
axs[1].hlines([np.sqrt(gamma_0 * 2)], xmin=-1, xmax=epochs*1.1, color=colors[0], linewidth=3, alpha=0.8, linestyle='--')
axs[1].hlines([np.sqrt(gamma_1 * 2)], xmin=-1, xmax=epochs*1.1, color=colors[1], linewidth=3, alpha=0.8, linestyle='--')

axs[1].set_yticks([0, 2.5, 5,])

y_pos = [text.get_position()[1] for text in axs[1].get_yticklabels()]#[:-1]
y_txt = [text.get_text() for text in axs[1].get_yticklabels()]#[:-1]

new_pos0 = np.sqrt(gamma_0 * 2)
new_txt0 = r"$\sqrt{2 \gamma_l}$"
new_pos1 = np.sqrt(gamma_1 * 2)
new_txt1 = r"$\sqrt{2 \gamma_s}$"

y_pos.append(new_pos0)
y_pos.append(new_pos1)
y_txt.append(new_txt0)
y_txt.append(new_txt1)
axs[1].set_yticks(y_pos,y_txt)

# axs[1].set_ylim(-0.0002, 0.0048)
axs[1].set_xlabel("Step")
axs[1].set_ylabel(r"$\frac{ds_j}{dt}$")
axs[0].legend()
axs[1].legend()
plt.tight_layout()
plt.savefig("figure/dim_s_ds_dt.pdf", format="pdf", bbox_inches = 'tight')
plt.show()

In [ ]:
#||A^TB||^2/2
fig, axs = plt.subplots(1, 1, figsize=(12, 4))
axs.plot(plane_alignment, alpha=0.8, color=colors[2], linewidth=3)
axs.set_xlim(0, epochs)
axs.set_ylim(-0.05, 1.05)
axs.set_xlabel("Step")
axs.set_ylabel("Subspace Alignment")
plt.savefig("figure/dim_alignment.pdf", format="pdf", bbox_inches = 'tight')
plt.show()
print(max(plane_alignment))

In [ ]:
#3d plot x: u1, y: u2, z: step, line1: We1, line2: We2
#use uwe[i][j], as uwe[0] means x, uwe[1] means z
def reject_outliers(data, m=0.04):
    indexes = np.where(abs(np.array(dt(data))) > m)
    for i in indexes:
        r = 10
        for j in range(-r, r):
            data[i+j] = data[i+j-1]
    return data

fig = plt.figure(figsize=(22, 10))
ax = fig.add_subplot(projection='3d')
for i in range(2):
    ax.plot(np.arange(0, epochs, 1), reject_outliers(np.abs(uwe[0][i])), reject_outliers(np.abs(uwe[1][i])), alpha=0.5, linewidth=5, color=colors[i], label=(r"$\text{Proj}_{U^{(\leq 2)}}(We_" + ("l" if i == 0 else "s") + ")$"))
ax.plot(np.arange(0, epochs, 1), np.sqrt(y1), 0, color=colors[0], linestyle='--')
ax.plot(np.arange(0, epochs, 1), 0, np.sqrt(y2), color=colors[1], linestyle='--')
ax.set_xlabel("Step", labelpad=6)
ax.set_ylabel(r"$u_1$", labelpad=6)
ax.set_zlabel(r"$u_2$", labelpad=6)
ax.set_box_aspect(aspect=None, zoom=0.98)
#center right
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.legend(loc='upper right', bbox_to_anchor=(1, 0.9))
plt.savefig("figure/dim_3d.pdf", format="pdf", bbox_inches = 'tight')
plt.show()